# python-jose

Smoke test for [python-jose](https://pypi.org/project/python-jose/): JSON Web Tokens signed with
HMAC and ECDSA keys, registered-claim validation, JSON Web Signatures, JSON Web Keys, and JSON Web
Encryption. Every signing key is generated at run time. Example 6 (JWE) needs AES support from the
`cryptography` extra: `pip install "python-jose[cryptography]"`.

## Example 1: Encode and decode a JWT with HS256

In [ ]:
import secrets
import time

from jose import jwt

signing_key = secrets.token_urlsafe(32)
now = int(time.time())
claims = {"sub": "1234567890", "name": "Jane Doe", "iat": now, "exp": now + 3600}

encoded = jwt.encode(claims, signing_key, algorithm="HS256")
print(f"JWT: {encoded[:40]}... ({len(encoded)} chars)")

# Compact serialisation: base64url(header).base64url(payload).base64url(signature)
assert encoded.count(".") == 2, encoded
assert jwt.get_unverified_header(encoded) == {"alg": "HS256", "typ": "JWT"}
assert jwt.get_unverified_claims(encoded) == claims

decoded = jwt.decode(encoded, signing_key, algorithms=["HS256"])
assert decoded == claims, decoded
print(f"decoded claims: sub={decoded['sub']} name={decoded['name']}")

## Example 2: HMAC algorithms and rejected tokens

In [ ]:
import base64
import json
import secrets
import time

from jose import jwt
from jose.exceptions import ExpiredSignatureError, JWTError

signing_key = secrets.token_urlsafe(32)
payload = {"user": "testuser", "role": "viewer", "iat": int(time.time())}

for alg in ("HS256", "HS384", "HS512"):
    encoded = jwt.encode(payload, signing_key, algorithm=alg)
    assert jwt.get_unverified_header(encoded) == {"alg": alg, "typ": "JWT"}
    assert jwt.decode(encoded, signing_key, algorithms=[alg]) == payload
    print(f"{alg}: {len(encoded)} chars, round trip OK")

sample = jwt.encode(payload, signing_key, algorithm="HS256")


def expect_rejected(description, encoded, key, algorithms, error=JWTError, **kwargs):
    try:
        jwt.decode(encoded, key, algorithms=algorithms, **kwargs)
    except error as exc:
        print(f"{description}: rejected ({type(exc).__name__}: {exc})")
    else:
        raise AssertionError(f"jwt.decode accepted a token with {description}")


expect_rejected("a different key", sample, secrets.token_urlsafe(32), ["HS256"])
expect_rejected("an algorithm that is not allowed", sample, signing_key, ["HS512"])

# Changing the payload invalidates the signature.
header_b64, _, signature_b64 = sample.split(".")
forged_payload = base64.urlsafe_b64encode(json.dumps({**payload, "role": "admin"}).encode()).rstrip(b"=").decode()
expect_rejected("a modified payload", f"{header_b64}.{forged_payload}.{signature_b64}", signing_key, ["HS256"])

# An unsigned token ("alg": "none") is never accepted when a key and algorithm list are given.
none_header = base64.urlsafe_b64encode(b'{"alg":"none","typ":"JWT"}').rstrip(b"=").decode()
expect_rejected("alg none", f"{none_header}.{forged_payload}.", signing_key, ["HS256"])

# Expiry: ExpiredSignatureError is a JWTError subclass; a small leeway accepts clock skew.
assert issubclass(ExpiredSignatureError, JWTError)
expired = jwt.encode({**payload, "exp": int(time.time()) - 3600}, signing_key, algorithm="HS256")
expect_rejected("an exp an hour ago", expired, signing_key, ["HS256"], error=ExpiredSignatureError)
just_expired = jwt.encode({**payload, "exp": int(time.time()) - 5}, signing_key, algorithm="HS256")
assert jwt.decode(just_expired, signing_key, algorithms=["HS256"], options={"leeway": 60})["user"] == "testuser"
print("a token 5 seconds past exp is accepted with a 60-second leeway")

## Example 3: Registered claims (aud, iss, sub, nbf) and custom headers

In [ ]:
import secrets
import time

from jose import jwt
from jose.exceptions import JWTClaimsError

signing_key = secrets.token_urlsafe(32)
now = int(time.time())
claims = {
    "iss": "https://issuer.example.com",
    "sub": "user:12345",
    "aud": "api.example.com",
    "iat": now,
    "nbf": now,
    "exp": now + 3600,
    "jti": secrets.token_hex(8),
    "scope": ["read", "write"],
}
encoded = jwt.encode(claims, signing_key, algorithm="HS256", headers={"kid": "key-001"})
assert jwt.get_unverified_header(encoded)["kid"] == "key-001"

decoded = jwt.decode(
    encoded,
    signing_key,
    algorithms=["HS256"],
    audience="api.example.com",
    issuer="https://issuer.example.com",
    subject="user:12345",
)
assert decoded == claims, decoded
print(f"validated: iss={decoded['iss']} aud={decoded['aud']} scope={decoded['scope']}")

checks = {
    "wrong audience": {"audience": "other.example.com"},
    "no audience given for a token that has one": {},
    "wrong issuer": {"audience": "api.example.com", "issuer": "https://other.example.com"},
    "wrong subject": {"audience": "api.example.com", "subject": "user:99999"},
}
for description, kwargs in checks.items():
    try:
        jwt.decode(encoded, signing_key, algorithms=["HS256"], **kwargs)
    except JWTClaimsError as exc:
        print(f"{description}: rejected ({exc})")
    else:
        raise AssertionError(f"jwt.decode accepted a token with {description}")

# A token is not valid before its nbf ("not before") time.
future = jwt.encode({"sub": "user:12345", "nbf": now + 3600}, signing_key, algorithm="HS256")
try:
    jwt.decode(future, signing_key, algorithms=["HS256"])
except JWTClaimsError as exc:
    print(f"nbf in the future: rejected ({exc})")
else:
    raise AssertionError("jwt.decode accepted a token before its nbf time")

## Example 4: JSON Web Signatures (JWS)

In [ ]:
import base64
import json
import secrets

from jose import jws
from jose.exceptions import JWSError

signing_key = secrets.token_urlsafe(32)
message = json.dumps({"message": "Test message", "user": "testuser"}).encode()

signed = jws.sign(message, signing_key, algorithm="HS384")
header = jws.get_unverified_header(signed)
assert header["alg"] == "HS384", header
assert jws.get_unverified_claims(signed) == message
assert jws.verify(signed, signing_key, algorithms=["HS384"]) == message
print(f"JWS: {signed[:40]}... header={header}")

# A dict payload is serialised to compact JSON before signing.
signed_dict = jws.sign({"m": 1}, signing_key, algorithm="HS256")
assert json.loads(jws.verify(signed_dict, signing_key, algorithms=["HS256"])) == {"m": 1}

# Replacing the payload breaks the signature.
header_b64, _, signature_b64 = signed.split(".")
other = base64.urlsafe_b64encode(b'{"message": "changed"}').rstrip(b"=").decode()
try:
    jws.verify(f"{header_b64}.{other}.{signature_b64}", signing_key, algorithms=["HS384"])
except JWSError as exc:
    print(f"tampered JWS rejected ({exc})")
else:
    raise AssertionError("jws.verify accepted a tampered payload")

## Example 5: ECDSA (ES256) tokens and JSON Web Keys

In [ ]:
import secrets

import ecdsa
from jose import jwk, jwt
from jose.exceptions import JWTError

# A P-256 key pair generated with the ecdsa package (a python-jose dependency), as PEM.
signing = ecdsa.SigningKey.generate(curve=ecdsa.NIST256p)
private_pem = signing.to_pem().decode()
public_pem = signing.get_verifying_key().to_pem().decode()

encoded = jwt.encode({"sub": "user:12345"}, private_pem, algorithm="ES256", headers={"kid": "ec-1"})
assert jwt.get_unverified_header(encoded) == {"alg": "ES256", "kid": "ec-1", "typ": "JWT"}
assert jwt.decode(encoded, public_pem, algorithms=["ES256"]) == {"sub": "user:12345"}

# The same public key as a JWK dict, alone and inside a JWK set.
public_jwk = jwk.construct(public_pem, algorithm="ES256").to_dict()
assert public_jwk["kty"] == "EC" and public_jwk["crv"] == "P-256", public_jwk
assert "d" not in public_jwk, "the public JWK contains the private scalar"
assert "d" in jwk.construct(private_pem, algorithm="ES256").to_dict()
public_jwk["kid"] = "ec-1"
assert jwt.decode(encoded, public_jwk, algorithms=["ES256"]) == {"sub": "user:12345"}
assert jwt.decode(encoded, {"keys": [public_jwk]}, algorithms=["ES256"]) == {"sub": "user:12345"}
print(f"ES256 verified with PEM, JWK and JWK set (x={public_jwk['x'][:12]}...)")

# Another key pair's public key does not verify the token.
stranger = ecdsa.SigningKey.generate(curve=ecdsa.NIST256p).get_verifying_key().to_pem().decode()
try:
    jwt.decode(encoded, stranger, algorithms=["ES256"])
except JWTError as exc:
    print(f"wrong public key rejected ({exc})")
else:
    raise AssertionError("an ES256 token verified with an unrelated public key")

# Symmetric keys are JWKs too: sign and verify raw bytes with an HMAC key.
hmac_key = jwk.construct(secrets.token_bytes(32), algorithm="HS256")
assert hmac_key.to_dict()["kty"] == "oct"
mac = hmac_key.sign(b"message")
assert hmac_key.verify(b"message", mac)
assert not hmac_key.verify(b"other message", mac)
print("HS256 JWK sign/verify OK")

## Example 6: JSON Web Encryption (JWE)

Needs `pip install "python-jose[cryptography]"`.

In [ ]:
import json
import secrets

import cryptography  # JWE needs python-jose's cryptography extra; fail early with a clear error
from jose import jwe
from jose.exceptions import JWEError

# Direct encryption with a 256-bit content key and AES-GCM.
content_key = secrets.token_bytes(32)
encrypted = jwe.encrypt(b"hello jwe", content_key, algorithm="dir", encryption="A256GCM")
# Compact JWE: header.encrypted_key.iv.ciphertext.tag
assert encrypted.count(b".") == 4, encrypted
header = jwe.get_unverified_header(encrypted)
assert header["alg"] == "dir" and header["enc"] == "A256GCM", header
assert jwe.decrypt(encrypted, content_key) == b"hello jwe"
print(f"dir + A256GCM: {len(encrypted)} bytes, decrypts OK")

# AES key wrap: a random content key is wrapped with a 128-bit key-encryption key.
wrapping_key = secrets.token_bytes(16)
document = json.dumps({"k": 1}).encode()
wrapped = jwe.encrypt(document, wrapping_key, algorithm="A128KW", encryption="A128CBC-HS256")
assert jwe.decrypt(wrapped, wrapping_key) == document
assert wrapped != jwe.encrypt(document, wrapping_key, algorithm="A128KW", encryption="A128CBC-HS256")
print("A128KW + A128CBC-HS256 round trip OK; each encryption is randomised")

try:
    jwe.decrypt(encrypted, secrets.token_bytes(32))
except JWEError as exc:
    print(f"wrong key rejected ({exc})")
else:
    raise AssertionError("jwe.decrypt accepted the wrong key")